In [52]:
!pip install chembl_webresource_client pandas rdkit scikit-learn openpyxl

In [53]:


!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
-f https://data.pyg.org/whl/torch-2.5.0+cpu.html

Looking in links: https://data.pyg.org/whl/torch-2.5.0+cpu.html


# LIBRARY IMPORTS

In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import warnings
warnings.filterwarnings("ignore")
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

# DATA INGESTION

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# ============================================
# FETCH ChEMBL ACTIVITY DATA
# ============================================

activity = new_client.activity

query = activity.filter(
    target_chembl_id="CHEMBL203",
    standard_type="IC50",
    standard_relation="="
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "assay_type",
    "assay_chembl_id",
    "confidence_score"
])

records = []

for i, rec in enumerate(query):

    records.append(rec)

    if i % 1000 == 0:
        print(f"Downloaded {i} records")

# ============================================
# CREATE DATAFRAME
# ============================================

data = pd.DataFrame(records)

print("Dataset Shape:", data.shape)

# ============================================
# SAVE TO EXCEL IN KAGGLE WORKING DIRECTORY
# ============================================

save_path = "/kaggle/working/EGFR_CHEMBL203_raw_data.xlsx"

data.to_excel(save_path, index=False)

print(f"\nExcel file saved at:\n{save_path}")

In [ ]:
print(data.columns)

# DATA CLEANING

In [ ]:
file_path = "/kaggle/working/EGFR_CHEMBL203_raw_data.xlsx"

data = pd.read_excel(file_path)

print("Original Shape:", data.shape)

# ============================================
# KEEP REQUIRED COLUMNS
# ============================================

required_columns = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "assay_type",
    "assay_chembl_id"
]

data = data[required_columns]

# ============================================
# REMOVE MISSING VALUES
# ============================================

data = data.dropna(
    subset=["canonical_smiles", "standard_value","assay_type"]
)

# ============================================
# CONVERT ACTIVITY VALUES TO NUMERIC
# ============================================

data["standard_value"] = pd.to_numeric(
    data["standard_value"],
    errors="coerce"
)

data = data.dropna(subset=["standard_value"])

print("After basic cleaning:", data.shape)

# STANDARDIZATION

In [ ]:
allowed_atoms = {
    "H", "C", "N", "O", "S",
    "P", "F", "Cl", "Br", "I"
}

normalizer = rdMolStandardize.Normalizer()
largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()


def standardize_smiles(smiles):

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        # Normalize
        mol = normalizer.normalize(mol)

        # Remove salts / keep largest fragment
        mol = largest_fragment_chooser.choose(mol)

        # Neutralize charges
        mol = uncharger.uncharge(mol)

        # Canonical tautomer normalization
        mol = tautomer_enumerator.Canonicalize(mol)

        # Molecular weight filtering
        mw = Descriptors.MolWt(mol)

        if mw < 100 or mw > 1000:
            return None

        # Allowed atom filtering
        atoms = {
            atom.GetSymbol()
            for atom in mol.GetAtoms()
        }

        if not atoms.issubset(allowed_atoms):
            return None

        # Canonical SMILES
        return Chem.MolToSmiles(
            mol,
            canonical=True
        )

    except:
        return None


# ============================================
# APPLY STANDARDIZATION
# ============================================

data["standardized_smiles"] = data[
    "canonical_smiles"
].apply(standardize_smiles)

# Remove failed molecules
data = data.dropna(
    subset=["standardized_smiles"]
)

print("After SMILES standardization:", data.shape)

# KEEP ONLY POSITIVE ACTIVITY VALUES


In [ ]:
data = data[
    data["standard_value"] > 0
]

print("After removing non-positive IC50:", data.shape)

# UNIT HARMONIZATION


In [ ]:

unit_conversion = {
    "nM": 1,
    "uM": 1000,
    "µM": 1000,
    "mM": 1_000_000
}


def convert_to_nM(row):

    unit = row["standard_units"]
    value = row["standard_value"]

    if unit in unit_conversion:
        return value * unit_conversion[unit]

    return np.nan


data["IC50_nM"] = data.apply(
    convert_to_nM,
    axis=1
)

data = data.dropna(
    subset=["IC50_nM"]
)

print("After unit harmonization:", data.shape)

# ============================================
# pIC50 CALCULATION
# ============================================

data["IC50_M"] = data["IC50_nM"] * 1e-9

data["pIC50"] = -np.log10(
    data["IC50_M"]
)

# ============================================
# BINARY ACTIVITY LABEL
# Active if pIC50 >= 6
# ============================================

data["activity_class"] = data[
    "pIC50"
].apply(
    lambda x: 1 if x >= 6 else 0
)

# DUPLICATE HANDLING

In [ ]:
from scipy.stats.mstats import gmean

def geometric_mean_pic50(values):

    values = np.array(values)

    # Convert pIC50 → IC50 molar
    ic50_values = 10 ** (-values)

    # Geometric mean IC50
    geo_ic50 = gmean(ic50_values)

    # Convert back → pIC50
    return -np.log10(geo_ic50)


data = (
    data.groupby(
        [
            "molecule_chembl_id",
            "standardized_smiles"
        ],
        as_index=False
    )
    .agg({
        "IC50_nM": lambda x: gmean(x),
        "pIC50": geometric_mean_pic50,
        "activity_class": "max",
        "assay_type": "first",
        "assay_chembl_id": "first"
    })
)

print("After duplicate handling:", data.shape)

# GENERATE MORGAN FINGERPRINTS
## For activity cliff detection

In [ ]:
# =====================================================
# GENERATE MORGAN FINGERPRINTS
# =====================================================

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

import random

print("\nGenerating fingerprints...")


def generate_fingerprint(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )


data["fingerprint"] = data[
    "standardized_smiles"
].apply(generate_fingerprint)

data = data.dropna(
    subset=["fingerprint"]
).reset_index(drop=True)

print(
    "After fingerprint generation:",
    data.shape
)


# =====================================================
# OPTIMIZED ACTIVITY CLIFF DETECTION
# Methodology:
# similarity >= 0.85
# delta_pIC50 >= 2
#
# Optimization:
# compare against random subset
# instead of full O(n²) search
# =====================================================

print("\nDetecting activity cliffs...")

activity_cliff_indices = set()

fingerprints = data["fingerprint"].tolist()
pic50_values = data["pIC50"].tolist()

num_molecules = len(data)

# ---------------------------------------------
# IMPORTANT OPTIMIZATION
# ---------------------------------------------
# Instead of comparing every pair,
# compare each molecule against
# a random subset
# ---------------------------------------------

MAX_COMPARISONS_PER_MOLECULE = 200

random.seed(42)

for i in range(num_molecules):

    # Create candidate indices excluding self
    candidate_indices = list(
        range(i + 1, num_molecules)
    )

    # Skip if no candidates left
    if len(candidate_indices) == 0:
        continue

    # Random subset sampling
    sampled_indices = random.sample(
        candidate_indices,
        min(
            MAX_COMPARISONS_PER_MOLECULE,
            len(candidate_indices)
        )
    )

    for j in sampled_indices:

        similarity = TanimotoSimilarity(
            fingerprints[i],
            fingerprints[j]
        )

        # Fast reject
        if similarity < 0.85:
            continue

        delta_pic50 = abs(
            pic50_values[i]
            - pic50_values[j]
        )

        if delta_pic50 >= 2:

            activity_cliff_indices.add(i)
            activity_cliff_indices.add(j)

    # Progress tracking
    if i % 1000 == 0:

        print(
            f"Processed "
            f"{i}/{num_molecules}"
        )


print(
    "\nActivity cliff molecules detected:",
    len(activity_cliff_indices)
)


# =====================================================
# ACTIVITY CLIFF FLAGGING
# =====================================================

print("\nFlagging activity cliffs...")

data["activity_cliff"] = False

if len(activity_cliff_indices) > 0:

    data.loc[
        list(activity_cliff_indices),
        "activity_cliff"
    ] = True


print("\nActivity Cliff Distribution:")

print(
    data["activity_cliff"]
    .value_counts()
)


# =====================================================
# REMOVAL
# =====================================================

REMOVE_ACTIVITY_CLIFFS = False

if REMOVE_ACTIVITY_CLIFFS:

    data = data[
        data["activity_cliff"] == False
    ].reset_index(drop=True)

    print(
        "\nAfter activity cliff removal:",
        data.shape
    )

# CLASS BALANCE ANALYSIS

In [ ]:
class_counts = data[
    "activity_class"
].value_counts()

print("\nClass Distribution:")
print(class_counts)

majority = class_counts.max()
minority = class_counts.min()

imbalance_ratio = majority / minority

print(
    f"\nClass imbalance ratio: "
    f"{imbalance_ratio:.2f}:1"
)


# =====================================================
# CLASS WEIGHTING DECISION
# Methodology:
# use class weighting if imbalance > 4:1
# =====================================================

print("\nCLASS WEIGHTING DECISION")

USE_CLASS_WEIGHTS = False

if imbalance_ratio > 4:

    USE_CLASS_WEIGHTS = True

    print(
        "WARNING: imbalance exceeds 4:1"
    )

    print(
        "Class weighting recommended."
    )

else:

    print(
        "Class balance acceptable."
    )

#  GENERATE MURCKO SCAFFOLDS

In [ ]:
def generate_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol
    )


data["scaffold"] = data[
    "standardized_smiles"
].apply(generate_scaffold)

data = data.dropna(
    subset=["scaffold"]
).reset_index(drop=True)

print(
    "After scaffold generation:",
    data.shape
)


# =====================================================
# SCAFFOLD FREQUENCY ANALYSIS
# =====================================================

print("\nSCAFFOLD FREQUENCY ANALYSIS")

scaffold_counts = (
    data["scaffold"]
    .value_counts()
)

sorted_scaffolds = (
    scaffold_counts.index.tolist()
)

print(
    "Unique scaffolds:",
    len(sorted_scaffolds)
)


# =====================================================
# FREQUENCY-AWARE
# SCAFFOLD SPLITTING
# =====================================================

print(
    "\n SCAFFOLD-BASED SPLITTING"
)

TRAIN_FRAC = 0.70
VALID_FRAC = 0.10
TEST_FRAC = 0.20

train_scaffolds = []
valid_scaffolds = []
test_scaffolds = []

train_count = 0
valid_count = 0
test_count = 0

total_molecules = len(data)

train_target = TRAIN_FRAC * total_molecules
valid_target = VALID_FRAC * total_molecules
test_target = TEST_FRAC * total_molecules


for scaffold in sorted_scaffolds:

    scaffold_size = scaffold_counts[
        scaffold
    ]

    if train_count < train_target:

        train_scaffolds.append(
            scaffold
        )

        train_count += scaffold_size

    elif valid_count < valid_target:

        valid_scaffolds.append(
            scaffold
        )

        valid_count += scaffold_size

    else:

        test_scaffolds.append(
            scaffold
        )

        test_count += scaffold_size

# CREATE SPLITS


In [ ]:
train_df = data[
    data["scaffold"].isin(
        train_scaffolds
    )
].reset_index(drop=True)

valid_df = data[
    data["scaffold"].isin(
        valid_scaffolds
    )
].reset_index(drop=True)

test_df = data[
    data["scaffold"].isin(
        test_scaffolds
    )
].reset_index(drop=True)


print("\nDATA SPLITS")

print(
    "Train:",
    train_df.shape
)

print(
    "Validation:",
    valid_df.shape
)

print(
    "Test:",
    test_df.shape
)


# =====================================================
# SCAFFOLD LEAKAGE CHECK
# =====================================================

print(
    "\nCHECKING "
    "SCAFFOLD LEAKAGE"
)

train_set = set(
    train_df["scaffold"]
)

valid_set = set(
    valid_df["scaffold"]
)

test_set = set(
    test_df["scaffold"]
)

assert len(
    train_set.intersection(valid_set)
) == 0

assert len(
    train_set.intersection(test_set)
) == 0

assert len(
    valid_set.intersection(test_set)
) == 0

print(
    "No scaffold leakage detected."
)


# =====================================================
# SCAFFOLD NOVELTY METRIC
# =====================================================

print(
    "\nSCAFFOLD "
    "NOVELTY METRIC"
)

unseen_test_scaffolds = (
    test_set - train_set
)

scaffold_novelty = (
    len(unseen_test_scaffolds)
    / len(test_set)
) * 100

print(
    f"Scaffold novelty "
    f"in test set: "
    f"{scaffold_novelty:.2f}%"
)


# =====================================================
#  REMOVE TEMP COLUMNS
# =====================================================

train_df = train_df.drop(
    columns=["fingerprint"]
)

valid_df = valid_df.drop(
    columns=["fingerprint"]
)

test_df = test_df.drop(
    columns=["fingerprint"]
)

In [ ]:
import os
print(
    "\nSAVING "
    "CLEANED DATASET"
)

cleaned_file = (
    "/kaggle/working/"
    "EGFR_CHEMBL203_cleaned_data.xlsx"
)

data.to_excel(
    cleaned_file,
    index=False
)

print(
    "Cleaned dataset saved:"
)

print(cleaned_file)


# =====================================================
# STEP 14 — SAVE SPLITS
# =====================================================

print(
    "\nSAVING SPLITS"
)

split_file = (
    "/kaggle/working/"
    "EGFR_CHEMBL203_split_data.xlsx"
)

with pd.ExcelWriter(
    split_file
) as writer:

    train_df.to_excel(
        writer,
        sheet_name="Train",
        index=False
    )

    valid_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )

print(
    "Split datasets saved:"
)

print(split_file)


# =====================================================
# STEP 15 — CLASS DISTRIBUTION
# ACROSS SPLITS
# =====================================================

print(
    "\nCLASS "
    "DISTRIBUTIONS"
)

print("\nTRAIN")
print(
    train_df[
        "activity_class"
    ].value_counts()
)

print("\nVALIDATION")
print(
    valid_df[
        "activity_class"
    ].value_counts()
)

print("\nTEST")
print(
    test_df[
        "activity_class"
    ].value_counts()
)


# =====================================================
# FILE CHECK
# =====================================================

print(
    "\nFILE CHECK"
)

print(
    os.listdir(
        "/kaggle/working/"
    )
)


# =====================================================
# FINAL SUMMARY
# =====================================================

print("\nFINAL SUMMARY")
print("=" * 50)

print(
    "Final dataset size:",
    len(data)
)

print(
    "Train molecules:",
    len(train_df)
)

print(
    "Validation molecules:",
    len(valid_df)
)

print(
    "Test molecules:",
    len(test_df)
)

print(
    "Activity cliff molecules:",
    len(activity_cliff_indices)
)

print(
    f"Imbalance ratio: "
    f"{imbalance_ratio:.2f}:1"
)

print(
    f"Scaffold novelty: "
    f"{scaffold_novelty:.2f}%"
)

print(
    "Use class weights:",
    USE_CLASS_WEIGHTS
)

# Graph representation

In [71]:
# ================================================================
# FULLY METHODOLOGY-ALIGNED GRAPH CONSTRUCTION
# + TRUE EDGE-AWARE MPNN (Gilmer-style)
#
# Fixes:
# ✅ larger atom vocabulary
# ✅ edge-aware message passing
# ✅ edge_attr actually used
# ✅ learned graph edges
# ✅ sparsification threshold
# ✅ edge weighting
# ✅ graph structure learning
# ✅ PyTorch Geometric format
# ✅ attention extraction support
# ================================================================

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    NNConv,
    global_mean_pool,
    global_max_pool,
    GATConv
)

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score
)

# ================================================================
# LOAD SPLITS
# ================================================================

split_file = "/kaggle/input/datasets/adwaithsmenon/egfr-chembl203-split-data/EGFR_CHEMBL203_split_data.xlsx"

train_df = pd.read_excel(
    split_file,
    sheet_name="Train"
)

valid_df = pd.read_excel(
    split_file,
    sheet_name="Validation"
)

test_df = pd.read_excel(
    split_file,
    sheet_name="Test"
)

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

SMILES_COL = "standardized_smiles"
LABEL_COL = "activity_class"

(7339, 9)
(1049, 9)
(2095, 9)


In [72]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

Using device: cuda


In [73]:
# ================================================================
# NODE FEATURE DEFINITIONS
# Methodology-aligned richer node features
# ================================================================

ATOM_TYPES = list(range(1, 45))

HYBRIDIZATION_TYPES = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

CHIRALITY_TYPES = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
]

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

STEREO_TYPES = [
    Chem.rdchem.BondStereo.STEREONONE,
    Chem.rdchem.BondStereo.STEREOZ,
    Chem.rdchem.BondStereo.STEREOE,
]


def one_hot(value, choices):

    encoding = [0] * len(choices)

    if value in choices:
        encoding[choices.index(value)] = 1

    return encoding

# ================================================================
# NODE FEATURES
# ================================================================

def atom_features(atom):

    features = []

    # atom type
    features += one_hot(
        atom.GetAtomicNum(),
        ATOM_TYPES
    )

    # degree
    features.append(atom.GetDegree())

    # formal charge
    features.append(atom.GetFormalCharge())

    # num hydrogens
    features.append(atom.GetTotalNumHs())

    # aromatic
    features.append(
        int(atom.GetIsAromatic())
    )

    # chirality
    features += one_hot(
        atom.GetChiralTag(),
        CHIRALITY_TYPES
    )

    # hybridization
    features += one_hot(
        atom.GetHybridization(),
        HYBRIDIZATION_TYPES
    )

    return features

In [74]:
# ================================================================
# EDGE FEATURES
# ================================================================

def bond_features(bond):

    features = []

    # bond type
    features += one_hot(
        bond.GetBondType(),
        BOND_TYPES
    )

    # stereo
    features += one_hot(
        bond.GetStereo(),
        STEREO_TYPES
    )

    # ring membership
    features.append(
        int(bond.IsInRing())
    )

    # conjugation
    features.append(
        int(bond.GetIsConjugated())
    )

    return features

In [75]:
# ================================================================
# CORRECTED GRAPH CONSTRUCTION
# Methodology-aligned
#
# FIXES:
# ✅ removes broken learned-edge logic
# ✅ keeps chemically meaningful edges
# ✅ keeps edge features
# ✅ keeps bidirectional edges
# ✅ compatible with edge-aware MPNN
# ================================================================

def smiles_to_graph(smiles, label):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    # ------------------------------------------------------------
    # NODE FEATURES
    # ------------------------------------------------------------

    node_features = []

    for atom in mol.GetAtoms():

        node_features.append(
            atom_features(atom)
        )

    x = torch.tensor(
        node_features,
        dtype=torch.float
    )

    # ------------------------------------------------------------
    # EDGE CONSTRUCTION
    # covalent bonds only
    # ------------------------------------------------------------

    edge_index = []

    edge_attr = []

    for bond in mol.GetBonds():

        start = bond.GetBeginAtomIdx()

        end = bond.GetEndAtomIdx()

        features = bond_features(
            bond
        )

        # forward edge
        edge_index.append(
            [start, end]
        )

        edge_attr.append(
            features
        )

        # reverse edge
        edge_index.append(
            [end, start]
        )

        edge_attr.append(
            features
        )

    # ------------------------------------------------------------
    # convert to tensors
    # ------------------------------------------------------------

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    edge_attr = torch.tensor(
        edge_attr,
        dtype=torch.float
    )

    # ------------------------------------------------------------
    # LABEL
    # ------------------------------------------------------------

    y = torch.tensor(
        [label],
        dtype=torch.float
    )

    # ------------------------------------------------------------
    # CREATE PYG GRAPH
    # ------------------------------------------------------------

    graph = Data(

        x=x,

        edge_index=edge_index,

        edge_attr=edge_attr,

        y=y
    )

    return graph

In [76]:
# ================================================================
# BUILD GRAPH DATASETS
# ================================================================

def build_graph_dataset(df):

    graphs = []

    for idx, row in df.iterrows():

        graph = smiles_to_graph(
            row[SMILES_COL],
            row[LABEL_COL]
        )

        if graph is not None:

            graphs.append(graph)

        if idx % 1000 == 0:

            print(
                f"Processed {idx}"
            )

    return graphs


train_graphs = build_graph_dataset(
    train_df
)

valid_graphs = build_graph_dataset(
    valid_df
)

test_graphs = build_graph_dataset(
    test_df
)

print(len(train_graphs))
print(len(valid_graphs))
print(len(test_graphs))

Processed 0
Processed 1000
Processed 2000
Processed 3000
Processed 4000
Processed 5000
Processed 6000
Processed 7000
Processed 0
Processed 1000
Processed 0
Processed 1000
Processed 2000
7339
1049
2095


In [77]:
# ================================================================
# DATALOADERS
# ================================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_graphs,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_graphs,
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    test_graphs,
    batch_size=BATCH_SIZE
)

In [78]:
class GraphStructureLearning(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.metric = nn.Linear(
            hidden_dim,
            hidden_dim
        )

    def forward(self, x):

        h = self.metric(x)

        sim = torch.matmul(
            h,
            h.T
        )

        adj = torch.softmax(
            sim,
            dim=-1
        )

        return torch.matmul(adj, x)

In [87]:
# ================================================================
# UPDATED METHODOLOGY-ALIGNED EDGE-AWARE MPNN
#
# Improvements:
# ✅ true edge-aware message passing
# ✅ deeper edge network
# ✅ residual connections
# ✅ Graph Structure Learning
# ✅ multi-head attention
# ✅ attention extraction support
# ✅ mean + max pooling
# ✅ batch norm
# ✅ dropout
# ================================================================

class EdgeAwareMPNN(nn.Module):

    def __init__(
        self,
        node_dim,
        edge_dim,
        hidden_dim=128
    ):

        super().__init__()

        # --------------------------------------------------------
        # NODE ENCODER
        # --------------------------------------------------------

        self.node_encoder = nn.Linear(
            node_dim,
            hidden_dim
        )

        # --------------------------------------------------------
        # EDGE NETWORK
        # deeper edge-conditioned network
        # --------------------------------------------------------

        edge_network = nn.Sequential(

            nn.Linear(
                edge_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                hidden_dim * hidden_dim
            )
        )

        # --------------------------------------------------------
        # EDGE-AWARE MESSAGE PASSING
        # --------------------------------------------------------

        self.conv1 = NNConv(

            hidden_dim,

            hidden_dim,

            edge_network,

            aggr="mean"
        )

        self.conv2 = NNConv(

            hidden_dim,

            hidden_dim,

            edge_network,

            aggr="mean"
        )

        # --------------------------------------------------------
        # GRAPH STRUCTURE LEARNING
        # --------------------------------------------------------

        self.gsl = GraphStructureLearning(
            hidden_dim
        )

        # --------------------------------------------------------
        # MULTI-HEAD ATTENTION
        # --------------------------------------------------------

        self.attention = GATConv(

            hidden_dim,

            hidden_dim,

            heads=8,

            concat=False,

            dropout=0.3
        )

        # --------------------------------------------------------
        # BATCH NORMALIZATION
        # --------------------------------------------------------

        self.bn1 = nn.BatchNorm1d(
            hidden_dim
        )

        self.bn2 = nn.BatchNorm1d(
            hidden_dim
        )

        # --------------------------------------------------------
        # PREDICTION HEAD
        # methodology:
        # 3-layer MLP
        # --------------------------------------------------------

        self.mlp = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                256
            ),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                128
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                1
            )
        )

    # ============================================================
    # FORWARD PASS
    # ============================================================

    def forward(self, data):

        x = data.x

        edge_index = data.edge_index

        edge_attr = data.edge_attr

        batch = data.batch

        # --------------------------------------------------------
        # INITIAL NODE EMBEDDING
        # --------------------------------------------------------

        x = self.node_encoder(x)

        # --------------------------------------------------------
        # GRAPH STRUCTURE LEARNING
        # --------------------------------------------------------

        x = self.gsl(x)

        # --------------------------------------------------------
        # FIRST MESSAGE PASSING LAYER
        # with residual connection
        # --------------------------------------------------------

        residual_1 = x

        x = self.conv1(

            x,

            edge_index,

            edge_attr
        )

        x = self.bn1(x)

        x = F.relu(x)

        x = x + residual_1

        # --------------------------------------------------------
        # SECOND MESSAGE PASSING LAYER
        # with residual connection
        # --------------------------------------------------------

        residual_2 = x

        x = self.conv2(

            x,

            edge_index,

            edge_attr
        )

        x = self.bn2(x)

        x = F.relu(x)

        x = x + residual_2

        # --------------------------------------------------------
        # MULTI-HEAD ATTENTION
        # --------------------------------------------------------

        x, attention_weights = self.attention(

            x,

            edge_index,

            return_attention_weights=True
        )

        # --------------------------------------------------------
        # SAVE ATTENTION WEIGHTS
        # for interpretability
        # --------------------------------------------------------

        self.latest_attention = (
            attention_weights
        )

        # --------------------------------------------------------
        # GLOBAL POOLING
        # methodology:
        # mean + max pooling
        # --------------------------------------------------------

        mean_pool = global_mean_pool(

            x,

            batch
        )

        max_pool = global_max_pool(

            x,

            batch
        )

        graph_embedding = torch.cat(

            [
                mean_pool,

                max_pool
            ],

            dim=1
        )

        # --------------------------------------------------------
        # FINAL PREDICTION
        # --------------------------------------------------------

        out = self.mlp(
            graph_embedding
        )

        return out.squeeze()

In [88]:
# ================================================================
# DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

cuda


In [89]:
# ================================================================
# MODEL INITIALIZATION
# ================================================================

sample_graph = train_graphs[0]

node_dim = sample_graph.x.shape[1]

edge_dim = sample_graph.edge_attr.shape[1]

model = EdgeAwareMPNN(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden_dim=128
).to(DEVICE)

print(model)

EdgeAwareMPNN(
  (node_encoder): Linear(in_features=56, out_features=128, bias=True)
  (conv1): NNConv(128, 128, aggr=mean, nn=Sequential(
    (0): Linear(in_features=9, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=16384, bias=True)
  ))
  (conv2): NNConv(128, 128, aggr=mean, nn=Sequential(
    (0): Linear(in_features=9, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=16384, bias=True)
  ))
  (gsl): GraphStructureLearning(
    (metric): Linear(in_features=128, out_features=128, bias=True)
  )
  (attention): GATConv(128, 128, heads=8)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (mlp): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  

In [90]:
# ================================================================
# LOSS
# ================================================================

criterion = nn.BCEWithLogitsLoss()

In [91]:
# ================================================================
# OPTIMIZER + COSINE ANNEALING
# ================================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)

In [92]:
# ================================================================
# TRAINING
# ================================================================

def train_epoch():

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(batch)

        loss = criterion(
            logits,
            batch.y.float()
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [93]:
# ================================================================
# EVALUATION
# ================================================================

def evaluate(loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for batch in loader:

            batch = batch.to(DEVICE)

            logits = model(batch)

            probs = torch.sigmoid(
                logits
            )

            preds.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                batch.y.cpu().numpy()
            )

    preds = np.array(preds)

    labels = np.array(labels)

    binary = (
        preds >= 0.5
    ).astype(int)

    return {

        "AUROC":
        roc_auc_score(
            labels,
            preds
        ),

        "AUPRC":
        average_precision_score(
            labels,
            preds
        ),

        "BalancedAccuracy":
        balanced_accuracy_score(
            labels,
            binary
        )
    }

In [94]:
# ================================================================
# TRAINING LOOP
# ================================================================

NUM_EPOCHS = 50

best_auroc = 0

patience = 6
counter = 0

save_path = (
    "/kaggle/working/"
    "best_edge_aware_mpnn.pt"
)

for epoch in range(NUM_EPOCHS):

    loss = train_epoch()

    val_metrics = evaluate(
        valid_loader
    )

    scheduler.step()

    val_auroc = (
        val_metrics["AUROC"]
    )

    val_auprc = (
        val_metrics["AUPRC"]
    )

    print(

        f"Epoch {epoch+1}/{NUM_EPOCHS} | "

        f"Loss: {loss:.4f} | "

        f"Val AUROC: {val_auroc:.4f} | "

        f"Val AUPRC: {val_auprc:.4f}"
    )

    if val_auroc > best_auroc:

        best_auroc = val_auroc

        torch.save(
            model.state_dict(),
            save_path
        )

        counter = 0

    else:

        counter += 1

    print(
        f"Patience Counter: {counter}"
    )

    if counter >= patience:

        print(
            "Early stopping triggered."
        )

        break

Epoch 1/50 | Loss: 0.4463 | Val AUROC: 0.7553 | Val AUPRC: 0.8915
Patience Counter: 0


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 2/50 | Loss: 0.3856 | Val AUROC: 0.7483 | Val AUPRC: 0.8890
Patience Counter: 1


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 3/50 | Loss: 0.3773 | Val AUROC: 0.7484 | Val AUPRC: 0.8927
Patience Counter: 2


...

Epoch 19/50 | Loss: 0.2574 | Val AUROC: 0.8038 | Val AUPRC: 0.9211
Patience Counter: 4


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 20/50 | Loss: 0.2593 | Val AUROC: 0.7857 | Val AUPRC: 0.9157
Patience Counter: 5


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 21/50 | Loss: 0.2507 | Val AUROC: 0.7948 | Val AUPRC: 0.9165
Patience Counter: 6
Early stopping triggered.


In [95]:
# ============================================================
# LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(save_path)
)

print("Best model loaded.")

Best model loaded.


In [96]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

test_metrics = evaluate(
    test_loader
)

print("\nTEST METRICS")
print("=" * 40)

for metric, value in test_metrics.items():

    print(
        f"{metric}: {value:.4f}"
    )


TEST METRICS
AUROC: 0.7968
AUPRC: 0.9078
BalancedAccuracy: 0.7339


In [97]:
# ============================================================
# SAVE TEST PREDICTIONS
# ============================================================

model.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(DEVICE)

        logits = model(batch)

        probs = torch.sigmoid(
            logits
        )

        preds = (
            probs >= 0.5
        ).float()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            batch.y.cpu().numpy()
        )


prediction_df = pd.DataFrame({

    "true_label": all_labels,

    "pred_probability": all_probs,

    "pred_label": all_preds
})

prediction_save_path = (

    "/kaggle/working/"
    "mpnn_test_predictions.csv"
)

prediction_df.to_csv(

    prediction_save_path,

    index=False
)

print(
    "\nPredictions saved:"
)

print(
    prediction_save_path
)


Predictions saved:
/kaggle/working/mpnn_test_predictions.csv


In [101]:
# ============================================================
# EXTRACT GRAPH EMBEDDINGS
# For meta learner stacking
# ============================================================

def extract_graph_embeddings(loader):

    model.eval()

    embeddings = []

    labels = []

    with torch.no_grad():

        for batch in loader:

            batch = batch.to(DEVICE)

            x = batch.x

            edge_index = batch.edge_index

            edge_attr = batch.edge_attr

            batch_index = batch.batch

            # ----------------------------------------------------
            # replicate forward pass up to graph embedding
            # ----------------------------------------------------

            x = model.node_encoder(x)

            x = model.gsl(x)

            residual_1 = x

            x = model.conv1(

                x,

                edge_index,

                edge_attr
            )

            x = model.bn1(x)

            x = F.relu(x)

            x = x + residual_1

            residual_2 = x

            x = model.conv2(

                x,

                edge_index,

                edge_attr
            )

            x = model.bn2(x)

            x = F.relu(x)

            x = x + residual_2

            x = model.attention(

                x,

                edge_index
            )

            # ----------------------------------------------------
            # graph pooling
            # ----------------------------------------------------

            mean_pool = global_mean_pool(

                x,

                batch_index
            )

            max_pool = global_max_pool(

                x,

                batch_index
            )

            graph_embedding = torch.cat(

                [
                    mean_pool,

                    max_pool
                ],

                dim=1
            )

            embeddings.extend(

                graph_embedding
                .cpu()
                .numpy()
            )

            labels.extend(

                batch.y
                .cpu()
                .numpy()
            )

    return (

        np.array(embeddings),

        np.array(labels)
    )

In [102]:
# ============================================================
# EXTRACT TEST EMBEDDINGS
# ============================================================

test_embeddings, test_labels = (

    extract_graph_embeddings(
        test_loader
    )
)

print(
    test_embeddings.shape
)

(2095, 256)


In [103]:
# ============================================================
# SAVE TEST EMBEDDINGS
# ============================================================

embedding_save_path = (

    "/kaggle/working/"
    "mpnn_test_embeddings.npy"
)

np.save(

    embedding_save_path,

    test_embeddings
)

print(
    "\nEmbeddings saved:"
)

print(
    embedding_save_path
)


Embeddings saved:
/kaggle/working/mpnn_test_embeddings.npy


In [104]:
# ============================================================
# SAVE EMBEDDINGS + LABELS CSV
# Useful for meta learner
# ============================================================

embedding_df = pd.DataFrame(
    test_embeddings
)

embedding_df["true_label"] = (
    test_labels
)

embedding_csv_path = (

    "/kaggle/working/"
    "mpnn_test_embeddings.csv"
)

embedding_df.to_csv(

    embedding_csv_path,

    index=False
)

print(
    "\nEmbedding CSV saved:"
)

print(
    embedding_csv_path
)


Embedding CSV saved:
/kaggle/working/mpnn_test_embeddings.csv
